In [ ]:
from pathlib import Path

import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ARTIFACT_DIR = Path("artifacts")

INPUT_PATH = ARTIFACT_DIR / "02_labeled_table.csv"

TRAIN_PATH = ARTIFACT_DIR / "03_train.csv"
VALIDATION_PATH = ARTIFACT_DIR / "03_validation.csv"
TEST_PATH = ARTIFACT_DIR / "03_test.csv"

SUMMARY_PATH = ARTIFACT_DIR / "03_split_summary.csv"


# ============================================================
# 2. VERIFY INPUT
# ============================================================

assert INPUT_PATH.exists(), (
    f"Input artifact not found: {INPUT_PATH}"
)

print(f"Reading artifact: {INPUT_PATH}")


# ============================================================
# 3. LOAD LABELED DATA
# ============================================================

df = pd.read_csv(
    INPUT_PATH,
    low_memory=False,
)

print("\nINPUT DATA")
print("=" * 80)

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")

assert "is_late" in df.columns
assert "order_purchase_timestamp" in df.columns
assert df["order_id"].is_unique


# ============================================================
# 4. PARSE ORDER PURCHASE DATE
# ============================================================

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    format="mixed",
    errors="coerce",
)

assert df["order_purchase_timestamp"].notna().all(), (
    "Some order_purchase_timestamp values could not be parsed."
)


# ============================================================
# 5. INSPECT DATE RANGE
# ============================================================

min_date = df["order_purchase_timestamp"].min()
max_date = df["order_purchase_timestamp"].max()

print("\nDATE RANGE")
print("=" * 80)

print(f"First order: {min_date}")
print(f"Last order:  {max_date}")


# ============================================================
# 6. SORT CHRONOLOGICALLY
# ============================================================

df = (
    df.sort_values(
        "order_purchase_timestamp"
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. TIME-BASED SPLIT
# ============================================================
#
# Train:      oldest 70%
# Validation: next 15%
# Test:       newest 15%
#
# Why:
# In production we train on historical orders and predict
# future orders. A chronological split better simulates
# that real deployment scenario.
# ============================================================

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

n_rows = len(df)

train_end = int(
    n_rows * TRAIN_RATIO
)

validation_end = int(
    n_rows * (
        TRAIN_RATIO
        + VALIDATION_RATIO
    )
)

train_df = (
    df.iloc[:train_end]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    df.iloc[
        train_end:validation_end
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    df.iloc[validation_end:]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 8. BASIC SPLIT VALIDATION
# ============================================================

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == len(df)
)

assert set(train_df["order_id"]).isdisjoint(
    validation_df["order_id"]
)

assert set(train_df["order_id"]).isdisjoint(
    test_df["order_id"]
)

assert set(validation_df["order_id"]).isdisjoint(
    test_df["order_id"]
)


# ============================================================
# 9. VALIDATE TEMPORAL ORDER
# ============================================================

train_max = (
    train_df["order_purchase_timestamp"]
    .max()
)

validation_min = (
    validation_df["order_purchase_timestamp"]
    .min()
)

validation_max = (
    validation_df["order_purchase_timestamp"]
    .max()
)

test_min = (
    test_df["order_purchase_timestamp"]
    .min()
)

assert train_max <= validation_min
assert validation_max <= test_min

print("\nTEMPORAL VALIDATION: PASS")


# ============================================================
# 10. SPLIT SUMMARY
# ============================================================

def build_split_summary(
    name: str,
    split_df: pd.DataFrame,
) -> dict[str, object]:

    total = len(split_df)

    late_count = int(
        split_df["is_late"].sum()
    )

    on_time_count = (
        total - late_count
    )

    late_percentage = (
        late_count
        / total
        * 100
    )

    return {
        "split": name,
        "rows": total,
        "percentage_of_dataset": (
            total / len(df) * 100
        ),
        "start_date": (
            split_df[
                "order_purchase_timestamp"
            ].min()
        ),
        "end_date": (
            split_df[
                "order_purchase_timestamp"
            ].max()
        ),
        "on_time_count": on_time_count,
        "late_count": late_count,
        "late_percentage": late_percentage,
    }


summary_df = pd.DataFrame(
    [
        build_split_summary(
            "train",
            train_df,
        ),
        build_split_summary(
            "validation",
            validation_df,
        ),
        build_split_summary(
            "test",
            test_df,
        ),
    ]
)


print("\nSPLIT SUMMARY")
print("=" * 80)

display(summary_df)


# ============================================================
# 11. CLASS DISTRIBUTION PER SPLIT
# ============================================================

for split_name, split_df in [
    ("TRAIN", train_df),
    ("VALIDATION", validation_df),
    ("TEST", test_df),
]:

    print(
        f"\n{split_name} CLASS DISTRIBUTION"
    )
    print("=" * 80)

    distribution = (
        split_df["is_late"]
        .value_counts()
        .sort_index()
        .rename_axis("is_late")
        .reset_index(name="count")
    )

    distribution["percentage"] = (
        distribution["count"]
        / len(split_df)
        * 100
    )

    display(distribution)


# ============================================================
# 12. IMPORTANT NOTE ABOUT CLASS RATIOS
# ============================================================
#
# We intentionally used a TIME-BASED split.
#
# Therefore we do NOT force each split to have exactly
# the same class ratio.
#
# Changes in late-delivery rate over time may be real
# distribution changes and are important to observe.
# ============================================================

print("\nSPLIT DECISION")
print("=" * 80)

print("Split strategy: TIME-BASED")

print(
    "Reason: production prediction is made "
    "for future orders using historical orders."
)

print(
    "Class ratios are not artificially forced "
    "to match across chronological splits."
)


# ============================================================
# 13. LEAKAGE REMINDER
# ============================================================

print("\nLEAKAGE REMINDER")
print("=" * 80)

print(
    "The test set must remain untouched "
    "during EDA, feature engineering, "
    "model selection, and tuning."
)

print(
    "Detailed EDA will use the TRAIN split only."
)


# ============================================================
# 14. SAVE ARTIFACTS
# ============================================================

train_df.to_csv(
    TRAIN_PATH,
    index=False,
)

validation_df.to_csv(
    VALIDATION_PATH,
    index=False,
)

test_df.to_csv(
    TEST_PATH,
    index=False,
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
)


# ============================================================
# 15. VERIFY SAVED ARTIFACTS
# ============================================================

for path in [
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
    SUMMARY_PATH,
]:
    assert path.exists()
    assert path.stat().st_size > 0


print("\nARTIFACTS")
print("=" * 80)

print(f"Train:      {TRAIN_PATH}")
print(f"Validation: {VALIDATION_PATH}")
print(f"Test:       {TEST_PATH}")
print(f"Summary:    {SUMMARY_PATH}")


# ============================================================
# 16. FINAL RESULT
# ============================================================

print("\n")
print("=" * 80)
print("NOTEBOOK 3 RESULT")
print("=" * 80)

print("SUCCESS")
print("A chronological train/validation/test split was created.")
print("No order appears in more than one split.")
print("Temporal ordering was validated.")
print("Class distribution was measured in each split.")
print("Train, validation, and test artifacts were saved.")